#### 清洗脏数据的部分就不多说了，这里主要是关于训练监督学习模型的方法和例子
#### 这里主要用到Linear Regression和RandomForestRegressor 两个模型，
#### 训练的数据是Kaggle的Medical Cost Personal Datasets

In [9]:
import pandas as pd
df = pd.read_csv('/Users/mr.tian/Desktop/insurance.csv')
df.columns = df.columns.str.title()
df = pd.get_dummies(df, columns=['Sex', 'Smoker', 'Region'], drop_first=True)
# 这里做了一些特征工程，添加了更具影响的列
df['Smoker_yes'] = df['Smoker_yes'].astype(int)
df['Bmi_smoker'] = df['Bmi'] * df['Smoker_yes']
df['Age_smoker'] = df['Age'] * df['Smoker_yes']

In [10]:
# 在这里选择了X和y的数据
X = df.drop(['Charges', 'Region_northwest', 'Region_southeast', 'Region_southwest'], axis=1)
y = df['Charges']

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

#### 这里运行交叉验证，交叉验证专注于模型表现，可以看到多次运行的r2得分
#### 如果要真正用模型做预测，仍需要训练模型步骤

In [27]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(scores)
print(scores.mean())

[0.86196568 0.79315533 0.87722952 0.81524467 0.83365984]
0.8362510077394534


#### 这里先运行一次标准化之前的

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)
r2_score(y_test, pred)

0.8631145226569426

#### 这里运行一次标准化之后的

In [21]:
scaler = StandardScaler()
scaler.fit(X_train) 
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

pred = model.predict(X_test_scaled)
r2_score(y_test, pred)

0.8629752898481591

#### 这里看一下特征重要性

In [22]:
importance = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(importance)

Bmi_smoker    18506.314855
Smoker_yes    -8510.191989
Age            3697.523684
Children        564.953288
Sex_male       -254.224069
Bmi              52.525901
Age_smoker       45.792512
dtype: float64


#### 这里做RandomForestRegressor的模型训练

In [23]:
model2 = RandomForestRegressor(n_estimators=500, random_state=42)
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
r2_score(y_test, pred2)

0.8617158784428579

In [26]:
# 这一步计算每个特征和目标y的线性相关系数，越大越相关
df.corr()['Charges'].sort_values(ascending=False)

Charges             1.000000
Bmi_smoker          0.845120
Age_smoker          0.789253
Smoker_yes          0.787251
Age                 0.299008
Bmi                 0.198341
Region_southeast    0.073982
Children            0.067998
Sex_male            0.057292
Region_northwest   -0.039905
Region_southwest   -0.043210
Name: Charges, dtype: float64

#### 换一组数据集，做监督学习的分类问题

In [28]:
df = pd.read_csv('/Users/mr.tian/Desktop/KaggleV2-May-2016.csv')
df = df.drop(columns=['Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap'])
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df = df[df['Age'] > 0]
df = pd.get_dummies(df, columns=['Gender', 'No-show'], drop_first=True)
cols = ['No-show_Yes', 'Gender_M']
df[cols] = df[cols].astype(int)

In [29]:
df['WaitingDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days
df.loc[df['WaitingDays'] < 0, 'WaitingDays'] = 0
df['sched_dow'] = df['ScheduledDay'].dt.dayofweek
df['appt_dow']  = df['AppointmentDay'].dt.dayofweek
df['sched_hour'] = df['ScheduledDay'].dt.hour
df['appt_hour']  = df['AppointmentDay'].dt.hour

In [35]:
X = df.drop(['No-show_Yes', 'ScheduledDay', 'AppointmentDay', 'Neighbourhood'], axis=1)
y = df['No-show_Yes']

#### 这里用到pipeline，且标准化，防止数据泄漏

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print('Accuracy_score: ',accuracy_score(y_test, pred))
print('F1_score: ',f1_score(y_test, pred))
print('Roc Auc: ',roc_auc_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy_score:  0.6564632208617628
F1_score:  0.3898065908524944
Roc Auc:  0.6140620843037498
              precision    recall  f1-score   support

           0       0.86      0.69      0.76     17073
           1       0.30      0.54      0.39      4325

    accuracy                           0.66     21398
   macro avg       0.58      0.61      0.58     21398
weighted avg       0.74      0.66      0.69     21398



#### 这下面的是模型训练的一个通用模版，可以作为参考

In [1]:
# 多模型模板 = “快速比较工具”
# 一次看清多个模型的准确率、召回率、F1，快速找到最优算法。 👇

In [7]:
# 模版
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 选取乳腺癌数据集，定义 X，y
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)

# 1. 分割数据
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 构建模型
models = {
    'Logistic': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'RandomForest': RandomForestClassifier(random_state=42)
}

# 3. 训练 + 评估
for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    print(f"{name}:",
          f"accuracy={accuracy_score(y_test, y_pred):.4f},",
          f"precision={precision_score(y_test, y_pred):.4f},",
          f"recall={recall_score(y_test, y_pred):.4f},",
          f"f1={f1_score(y_test, y_pred):.4f}")


Logistic: accuracy=0.9737, precision=0.9722, recall=0.9859, f1=0.9790
KNN: accuracy=0.9474, precision=0.9577, recall=0.9577, f1=0.9577
RandomForest: accuracy=0.9649, precision=0.9589, recall=0.9859, f1=0.9722


In [ ]:
# 聚焦单个模型（如 Logistic Regression），
# 从数据准备 → 训练 → 测试 → 单样本预测 → 概率输出，全流程演示。 👇

In [ ]:
# 数据集：乳腺癌（Breast Cancer）
# 模型：逻辑回归（可随时切换其他模型）

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2️⃣ 加载数据并划分
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 3️⃣ 数据标准化（非常关键！特别是对 SVM / KNN / Logistic 等模型）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4️⃣ 导入常见模型（只导入，不一定都用）
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# 5️⃣ 选择模型（可更换成上面任意一个）
clf = LogisticRegression()                                     # 逻辑回归模型

# 6️⃣ 模型训练
clf.fit(X_train_scaled, y_train)                               # 拟合训练集

# 7️⃣ 模型评估
clf.score(X_test_scaled, y_test)                               # 输出测试集准确率

# 8️⃣ 单样本预测
single_instance = X_test[1]                                   # 取一个测试样本
clf.predict([single_instance])                                 # 预测该样本类别
y_test[1]                                                      # 实际类别（对照）

# 9️⃣ 输出预测概率（属于每个类别的概率）
clf.predict_proba([single_instance])                  # e.g. [[0.03, 0.97]] → 预测为第二类概率97%